<a href="https://colab.research.google.com/github/angelicavale9220-ship-it/Unicafam_Modelos_Analiticos_Indra/blob/main/Sesion3_Angelica_m__Ejercicios_integracion_procesamiento_de_datos_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***CLIENTES INACTIVOS***

In [157]:
# CLIENTES INACTIVOS

import pandas as pd

In [162]:
# 1 Clientes nuevos

clientes = pd.DataFrame({

    "id_cliente": ["C001", "C002", "C003", "C004", "C005"],
    "nombre": ["Luis Peña", "Juan Ríos", "Sofía Mora",
               "Carlos Ruiz", "María López"],
    "ciudad": ["Bogotá", "Cali", "Medellín", "Cali", "Bogotá"],
    "segmento": ["Bronce", "Plata", "Oro", "Bronce", "Oro"]
})

In [163]:
# PEDIDOS

pedidos = pd.DataFrame({

    "pedido_id": [101, 102, 103, 104],
    "cliente": ["C004", "C005", "C009", "C010"]
})

In [164]:
#NUEVOS

nuevos = pd.DataFrame({

    "id_cliente": ["C006", "C007", "C008"],
    "nombre": ["Pablo Díaz", "Eva Soto", "Iván Cruz"],
    "ciudad": ["Cali", "Bogotá", "Medellín"],
    "segmento": ["Bronce", "Oro", "Plata"]
})

clientes2 = pd.concat([clientes, nuevos], ignore_index=True)

In [165]:
#MERGE

m = pd.merge(

    clientes2,
    pedidos,
    left_on="id_cliente",
    right_on="cliente",
    how="outer",
    indicator=True
)


In [167]:
print(m["_merge"].value_counts().to_dict())

print("Campaña:")

print(m.loc[m["_merge"] == "left_only", "nombre"].to_string(index=False))

{'left_only': 6, 'right_only': 2, 'both': 2}
Campaña:
 Luis Peña
 Juan Ríos
Sofía Mora
Pablo Díaz
  Eva Soto
 Iván Cruz


# ***CONSOLIDADO TRIMESTRAL***

In [168]:
# CONSOLIDADO

import pandas as pd

In [169]:
# DATOS

datos = {
    "sucursal": [
        "Centro", "Centro",
        "Sur", "Sur",
        "Norte", "Norte"
    ],
    "mes": [
        "Enero", "Marzo",
        "Enero", "Marzo",
        "Enero", "Marzo"
    ],
    "ventas": [
        1000, 1565,
        1000, 850,
        1000, 726
    ]
}

In [170]:
# CONSOLIDADO

consolidado = pd.DataFrame(datos)

In [171]:
t = consolidado.pivot_table(
    index="sucursal",
    columns="mes",
    values="ventas"
)

t["crecimiento_%"] = (
    (t["Marzo"] - t["Enero"]) /
    t["Enero"] * 100
).round(1)

In [172]:
t = t.sort_values("crecimiento_%", ascending=False)

In [173]:
print(t["crecimiento_%"])

print("Líder:", t.index[0])

sucursal
Centro    56.5
Sur      -15.0
Norte    -27.4
Name: crecimiento_%, dtype: float64
Líder: Centro


# ***SQL VS PANDAS***

In [250]:
import pandas as pd
import sqlite3

In [251]:
# 1. Datos de ejemplo (para que Tarjeta sume 360000)
pedidos = pd.DataFrame({
    "pago.medio": ["Tarjeta", "Efectivo", "Tarjeta", "Transferencia", "Tarjeta"],
    "pago.valor": [150000, 80000, 110000, 95000, 100000]
})


In [252]:
# 2. Conexión a SQLite en memoria
con = sqlite3.connect(":memory:")

In [253]:
# 3. Resultado con pandas
por_medio = pedidos.groupby("pago.medio")["pago.valor"].sum()
print("pandas:", por_medio.idxmax(), por_medio.max())

pandas: Tarjeta 360000


In [254]:
# 4. Resultado con SQL
pedidos.rename(columns={"pago.medio": "medio", "pago.valor": "valor"}).to_sql(
    "pagos", con, index=False, if_exists="replace"
)
q = """SELECT medio, SUM(valor) AS total FROM pagos
       GROUP BY medio ORDER BY total DESC LIMIT 1"""
print("SQL:", pd.read_sql(q, con).values.tolist())

SQL: [['Tarjeta', 360000]]


In [256]:
import pandas as pd
import sqlite3

# 1. Creamos la tabla pedidos con sus valores
pedidos = pd.DataFrame({
    "pago.medio": ["Tarjeta", "Efectivo", "Tarjeta", "Transferencia", "Tarjeta"],
    "pago.valor": [150000, 80000, 110000, 90000, 100000]
})

# 2. Creamos la conexión a SQLite
con = sqlite3.connect(":memory:")

# 3. Cálculo con pandas
por_medio = pedidos.groupby("pago.medio")["pago.valor"].sum()
print("pandas:", por_medio.idxmax(), por_medio.max())

# 4. Cálculo con SQL
pedidos.rename(columns={"pago.medio": "medio", "pago.valor": "valor"}).to_sql(
    "pagos", con, index=False, if_exists="replace"
)
q = """SELECT medio, SUM(valor) AS total FROM pagos
       GROUP BY medio ORDER BY total DESC LIMIT 1"""
print("SQL:", pd.read_sql(q, con).values.tolist())

pandas: Tarjeta 360000
SQL: [['Tarjeta', 360000]]
